# C1 · Notebook 02 — Cleaning and analytic table (2014–2024)
**Business Intelligence · IIB423T-1 · UDD** — **Group 06** — Maximiliano Sandoval and Benjamín Toloza

**Purpose.** Turn the raw tables into **one analytic table** in which **one row = one driver in one race (2014–2024)**, with the
variables needed by the KPIs and by the EDA in notebook 03. Every cleaning rule comes from a finding in notebook 01 and is
**checked** (row counts, key uniqueness, join validation) instead of assumed. At the end, a cleaning log lists each step and its effect.

| Input (data/raw/) | Output (data/processed/) |
|---|---|
| races, results, qualifying, lap_times, pit_stops, constructor_standings, drivers, constructors, circuits, status, sprint_results | `analytic_driver_race_2014_2024.csv` + `cleaning_log.csv` + `data_dictionary.csv` |

## 0. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from pathlib import Path

def find_root():
    """Finds the project folder (the one containing data/raw/races.csv), starting from where the
    notebook runs and moving up one folder at a time. Works the same in Jupyter and in VS Code."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "races.csv").exists():
            return base
    raise FileNotFoundError("data/raw/races.csv not found. Open the whole GROUP_06_C1 folder "
                            "(File > Open Folder) and check that the Kaggle CSVs are in data/raw/.")

ROOT = find_root()
RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "analysis" / "figures"
for d in (PROC, FIG):
    os.makedirs(d, exist_ok=True)

NA = "\\N"                                   # how missing values are written in this dataset
read = lambda file: pd.read_csv(RAW / file, na_values=NA)

# Plot style: one colour per series, thin marks, light grid, text in grey (never in the series colour)
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e6e3"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])   # red (-) · grey (0) · blue (+)
plt.rcParams.update({
    "figure.figsize": (8, 4.2), "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#bdbcb7",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "font.size": 9.5,
    "legend.frameon": False})

def save_fig(fig, name):
    """Saves the figure in analysis/figures/ (used by the report) and shows it in the notebook."""
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def summary(s):
    """Classical and robust summary of a numeric variable (mean/SD vs median/IQR, shape)."""
    s = pd.Series(s).dropna()
    return pd.Series({"n": len(s), "mean": s.mean(), "std": s.std(), "min": s.min(), "p25": s.quantile(.25),
                      "median": s.median(), "p75": s.quantile(.75), "max": s.max(),
                      "IQR": s.quantile(.75) - s.quantile(.25), "skew": s.skew(), "kurtosis": s.kurt()})

print("Project folder:", ROOT.name, "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scipy", __import__("scipy").__version__, "| matplotlib", matplotlib.__version__)

Project folder: GROUP_06_C1 | pandas 3.0.2 | numpy 2.4.4 | scipy 1.17.1 | matplotlib 3.10.9


## 1. Master filter: seasons 2014–2024
`races.csv` defines the period once; every fact table is then trimmed to the same `raceId`s, so all variables describe exactly
the same population of races. Reasons for the period (notebook 01, sections 3–4): one engine formula, one points scale,
knockout qualifying, and every table needed is complete.

In [2]:
PERIOD = (2014, 2024)
LOG = []                                   # cleaning log: one entry per consequential step
def log(step, rule, before, after, evidence):
    LOG.append({"step": step, "rule": rule, "rows before": before, "rows after": after, "evidence": evidence})

races_all = read("races.csv")
races = races_all[races_all["year"].between(*PERIOD)][["raceId", "year", "round", "circuitId", "name", "date"]].copy()
ids = set(races["raceId"])

def trim(file):
    df = read(file)
    out = df[df["raceId"].isin(ids)].copy()
    print(f"{file:<26} {len(df):>8,} rows -> {len(out):>7,} in {PERIOD[0]}–{PERIOD[1]}")
    return out

results     = trim("results.csv")
qualifying  = trim("qualifying.csv")
lap_times   = trim("lap_times.csv")
pit_stops   = trim("pit_stops.csv")
sprint      = trim("sprint_results.csv")
cstand_all  = read("constructor_standings.csv")          # kept whole: round 1 needs the previous season
drivers, constructors, circuits, status = read("drivers.csv"), read("constructors.csv"), read("circuits.csv"), read("status.csv")
log("1. Period", "Keep seasons 2014–2024 (master filter on races.raceId)", len(read("results.csv")), len(results),
    "Notebook 01, parts 3–4")
print(f"\nRaces: {len(races)} | seasons: {races['year'].nunique()} | circuits: {races['circuitId'].nunique()}")

results.csv                  26,759 rows ->   4,626 in 2014–2024
qualifying.csv               10,494 rows ->   4,610 in 2014–2024
lap_times.csv               589,081 rows -> 248,144 in 2014–2024
pit_stops.csv                11,371 rows ->   8,360 in 2014–2024
sprint_results.csv              360 rows ->     360 in 2014–2024

Races: 228 | seasons: 11 | circuits: 32


## 2. Base table from `results` and its outcome variables
`results` already has the target unit (driver-race). We check its key and derive the outcome variables:

- `finished`: status "Finished" or "+n Laps" (the car took the chequered flag).
- `classified`: `position` not empty (a driver who retires after ≥ 90 % of the distance is still classified).
- `final_pos` = `positionOrder` (complete order, retirements at the bottom). It is the outcome used for the KPIs among finishers,
  where it coincides with `position`.

In [3]:
assert not results.duplicated(["raceId", "driverId"]).any(), "duplicated driver-race in results"
df = (results[["raceId", "driverId", "constructorId", "grid", "position", "positionOrder", "points", "laps", "statusId", "rank"]]
      .merge(races, on="raceId", how="left", validate="many_to_one")
      .merge(status, on="statusId", how="left", validate="many_to_one"))
df = df.rename(columns={"positionOrder": "final_pos", "rank": "fastest_lap_rank"})
print("fastest_lap_rank = 0 (no timed lap) -> NaN:", int((df["fastest_lap_rank"] == 0).sum()))
df.loc[df["fastest_lap_rank"] == 0, "fastest_lap_rank"] = np.nan
df["finished"] = df["status"].eq("Finished") | df["status"].str.match(r"^\+\d+ Laps?$")
df["classified"] = df["position"].notna()

# Check: positionOrder is a complete ranking 1..n inside every race
ok = df.groupby("raceId")["final_pos"].apply(lambda s: sorted(s) == list(range(1, len(s) + 1)))
print(f"Races where final_pos is exactly 1..n: {ok.mean():.0%}")
print(f"Finished: {df['finished'].mean():.1%} | classified: {df['classified'].mean():.1%} | "
      f"classified without finishing: {(df['classified'] & ~df['finished']).sum()}")

# Retirement cause, grouped (139 raw statuses are too many to read)
def cause(s):
    if s == "Finished" or pd.Series([s]).str.match(r"^\+\d+ Laps?$").iat[0]: return "finished"
    if s in ("Accident", "Collision", "Collision damage", "Spun off", "Damage", "Debris", "Puncture", "Tyre"): return "incident"
    if s in ("Disqualified", "Excluded", "Withdrew", "Did not qualify", "Illness", "Injured", "Injury"): return "other (DSQ/withdrew)"
    return "mechanical / technical"
df["end_cause"] = df["status"].map(cause)
df["end_cause"].value_counts()

fastest_lap_rank = 0 (no timed lap) -> NaN: 217
Races where final_pos is exactly 1..n: 100%
Finished: 82.7% | classified: 84.4% | classified without finishing: 81


end_cause
finished                  3825
mechanical / technical     453
incident                   319
other (DSQ/withdrew)        29
Name: count, dtype: int64

**Reading.** The key (race, driver) is unique and `final_pos` is a complete ranking 1..n in every race, so it can be used as an
ordinal outcome without gaps. `fastest_lap_rank = 0` is another special code (no timed lap) and becomes `NaN`. 82.7 % of results finished; 81 drivers were classified without finishing, which is why
"finished" and "classified" are kept as separate flags. The 139 raw statuses are grouped into 4 causes for readable plots.

## 3. Races that do not measure the question
Belgium 2021 was flagged in W1. Here the rule is **data-driven**: a race is excluded if its lap chart has fewer than 5 laps
(no real racing, so the finishing order just copies the grid and ρ = 1 by construction).

In [4]:
laps_per_race = lap_times.groupby("raceId")["lap"].max()
not_raced = laps_per_race[laps_per_race < 5].index
display(races[races["raceId"].isin(not_raced)].assign(laps_recorded=lambda d: d["raceId"].map(laps_per_race)))
n0 = len(df)
df = df[~df["raceId"].isin(not_raced)].copy()
log("2. Not-raced races", "Exclude races with < 5 laps in lap_times", n0, len(df), "Belgium 2021: 1 lap behind the safety car")
print(f"Rows: {n0} -> {len(df)} | races: {df['raceId'].nunique()}")

,raceId,year,round,circuitId,name,date,laps_recorded
1048,1063,2021,12,13,Belgian Grand Prix,2021-08-29,1


Rows: 4626 -> 4606 | races: 227


## 4. Starting position: pit-lane starts and grid blocks
`grid = 0` means **pit-lane start** in this era (notebook 01, part 5). It is not a position smaller than 1, so:
- `pitlane_start` flags it;
- `grid` becomes `NaN` for those rows (rank statistics must not treat it as "position 0");
- `grid_block` groups positions for the conditional KPIs (P(top 10 | block)), with "pit lane" as its own group.

`positions_gained = grid − final_pos` (positive = the driver gained places).

In [5]:
df["pitlane_start"] = df["grid"].eq(0)
df.loc[df["pitlane_start"], "grid"] = np.nan
df["grid_block"] = pd.cut(df["grid"], [0, 3, 6, 10, 15, 30], labels=["1–3", "4–6", "7–10", "11–15", "16+"]).astype(object)
df.loc[df["pitlane_start"], "grid_block"] = "pit lane"
df["positions_gained"] = df["grid"] - df["final_pos"]
print(f"Pit-lane starts: {df['pitlane_start'].sum()} ({df['pitlane_start'].mean():.1%}) | grid range: "
      f"{df['grid'].min():.0f}–{df['grid'].max():.0f}")
df["grid_block"].value_counts().reindex(["1–3", "4–6", "7–10", "11–15", "16+", "pit lane"])

Pit-lane starts: 77 (1.7%) | grid range: 1–22


grid_block
1–3          679
4–6          681
7–10         905
11–15       1128
16+         1136
pit lane      77
Name: count, dtype: int64

## 5. Qualifying: position and pace
**Join** `results → qualifying` on (raceId, driverId), expected one-to-one; tested with `validate` and `indicator`.

**Qualifying pace.** Times come as text (`"1:26.572"`) and must be converted to seconds. For each driver we take the best of
Q1/Q2/Q3 and express it as a **gap to the fastest driver of that race, in %**. A percentage makes circuits of different lengths
comparable (1 s at Monaco ≠ 1 s at Spa).

*Limitation recorded:* if one session was wet and another dry, the best time mixes conditions (no weather data in the dataset).
Gaps above 7 % (the classic 107 % rule) are flagged as outliers.

In [6]:
def to_seconds(t):
    """'1:26.572' -> 86.572 ; NaN or malformed -> NaN"""
    if pd.isna(t) or ":" not in str(t):
        return np.nan
    m, s = str(t).split(":")
    return int(m) * 60 + float(s)

for c in ["q1", "q2", "q3"]:
    qualifying[c + "_s"] = qualifying[c].map(to_seconds)
qualifying["quali_best_s"] = qualifying[["q1_s", "q2_s", "q3_s"]].min(axis=1)
qualifying["quali_gap_pct"] = 100 * (qualifying["quali_best_s"] / qualifying.groupby("raceId")["quali_best_s"].transform("min") - 1)
print("Unparseable/missing q1 times:", int(qualifying["q1_s"].isna().sum()))

n0 = len(df)
df = df.merge(qualifying[["raceId", "driverId", "position", "quali_best_s", "quali_gap_pct"]]
              .rename(columns={"position": "quali_pos"}),
              on=["raceId", "driverId"], how="left", validate="one_to_one", indicator=True)
print("Join results -> qualifying:", df["_merge"].value_counts().to_dict())
df = df.drop(columns="_merge"); assert len(df) == n0

df["quali_outlier"] = df["quali_gap_pct"] > 7
df["grid_differs_quali"] = df["grid"].notna() & df["quali_pos"].notna() & (df["grid"] != df["quali_pos"])
print(f"Gap > 7 % (flagged): {df['quali_outlier'].sum()} | grid differs from qualifying: {df['grid_differs_quali'].mean():.1%}")
display(summary(df["quali_gap_pct"]).round(2).to_frame("quali gap to pole (%)"))

Unparseable/missing q1 times: 51
Join results -> qualifying: {'both': 4590, 'left_only': 16, 'right_only': 0}
Gap > 7 % (flagged): 105 | grid differs from qualifying: 29.6%


,quali gap to pole (%)
n,4539.00
mean,2.23
std,2.46
min,0.00
p25,0.94
median,1.85
p75,2.85
max,45.60
IQR,1.91
skew,6.23


**Reading.** The join is one-to-one and almost complete; the few rows without qualifying are drivers who set no time or were
excluded. In ~30 % of results the grid differs from the qualifying position (penalties, sprint-based grids in 2021–22),
so notebook 03 treats "start → result" as the main question and "qualifying → result" as a sensitivity check.
The gap to the fastest qualifier is strongly right-skewed (median 1.85 %, IQR 0.9–2.9 %, max 45.6 %): most of the field
is within 1–3 %, with a long tail of drivers who had problems or set their best time in a wet session (105 rows above 7 %, flagged).

## 6. First lap and race pace (from `lap_times` and `pit_stops`)
- `pos_lap1`: position at the end of lap 1. It separates **the start** (grid → lap 1) from **the race** (lap 1 → finish).
- `race_pace_gap_pct`: a driver's typical racing lap relative to the fastest driver of that race, in %. To measure *pace* and not
  incidents, we remove lap 1, **in- and out-laps of pit stops**, and laps slower than 107 % of the race's median lap (safety car,
  red flag, damage). The driver's pace is the **median** of the remaining laps (robust to outliers) and is computed only for
  drivers with at least 50 % of the race's laps.

In [7]:
lap1 = lap_times[lap_times["lap"] == 1][["raceId", "driverId", "position"]].rename(columns={"position": "pos_lap1"})

lt = lap_times[~lap_times["raceId"].isin(not_raced)].copy()
pit_laps = pd.concat([pit_stops[["raceId", "driverId", "lap"]],                                   # in-lap
                      pit_stops[["raceId", "driverId", "lap"]].assign(lap=lambda d: d["lap"] + 1)])  # out-lap
pit_laps["pit_lap"] = True
n_laps = len(lt)
lt = lt.merge(pit_laps.drop_duplicates(), on=["raceId", "driverId", "lap"], how="left", validate="many_to_one")
assert len(lt) == n_laps                       # flagging pit laps must not duplicate any lap
race_median = lt.groupby("raceId")["milliseconds"].transform("median")
lt["green"] = (lt["lap"] > 1) & lt["pit_lap"].isna() & (lt["milliseconds"] <= 1.07 * race_median)
print(f"Laps kept as representative racing laps: {lt['green'].mean():.1%} of {len(lt):,}")

pace = lt[lt["green"]].groupby(["raceId", "driverId"])["milliseconds"].agg(pace_ms="median", green_laps="size").reset_index()
race_laps = lt.groupby("raceId")["lap"].max().rename("race_laps")
pace = pace.merge(race_laps, on="raceId")
pace = pace[pace["green_laps"] >= 0.5 * pace["race_laps"]]
pace["race_pace_gap_pct"] = 100 * (pace["pace_ms"] / pace.groupby("raceId")["pace_ms"].transform("min") - 1)

n0 = len(df)
df = (df.merge(lap1, on=["raceId", "driverId"], how="left", validate="one_to_one")
        .merge(pace[["raceId", "driverId", "race_pace_gap_pct"]], on=["raceId", "driverId"], how="left", validate="one_to_one"))
assert len(df) == n0
df["gained_lap1"] = df["grid"] - df["pos_lap1"]
print(f"pos_lap1 available: {df['pos_lap1'].notna().mean():.1%} | race pace available: {df['race_pace_gap_pct'].notna().mean():.1%} "
      f"(finishers: {df.loc[df['finished'], 'race_pace_gap_pct'].notna().mean():.1%})")
display(summary(df["race_pace_gap_pct"]).round(2).to_frame("race pace gap (%)"))

Laps kept as representative racing laps: 85.8% of 248,124
pos_lap1 available: 97.0% | race pace available: 88.7% (finishers: 99.9%)


,race pace gap (%)
n,4087.00
mean,1.69
std,1.16
min,0.00
p25,0.78
median,1.67
p75,2.42
max,11.32
IQR,1.64
skew,0.82


**Reading.** Both joins are one-to-one and keep the row count. Lap 1 is missing only for drivers who did not complete it.
Race pace is available for practically every finisher and missing, by design, for early retirements (too few laps to judge pace).
Its distribution is tighter than the qualifying gap: in the race, the whole field usually sits within ~1–3 % of the fastest car.

## 7. Team strength before the race (control variable) and team lineage
**Why:** fast cars qualify at the front *and* finish at the front, so part of the start → result relationship is really the car.
**Measure:** the team's championship position **after the previous round** (`constructor_rank_prev`). Using the standings after the
same race would include the result we want to explain (**leakage**).
- Round 1 has no previous round → we use the **final standings of the previous season**, matched by team **lineage**,
  because teams change name and id between seasons (e.g. Racing Point → Aston Martin).
- The lineage map (W1, section A.5) also gives stable team names for plots.

In [8]:
LINEAGE = {10: "Aston Martin", 211: "Aston Martin", 117: "Aston Martin",           # Force India -> Racing Point -> Aston Martin
           5: "RB", 213: "RB", 215: "RB",                                          # Toro Rosso -> AlphaTauri -> RB
           15: "Sauber", 51: "Sauber",                                             # Sauber -> Alfa Romeo
           208: "Alpine", 4: "Alpine", 214: "Alpine",                              # Lotus F1 -> Renault -> Alpine
           206: "Manor", 209: "Manor"}                                             # Marussia -> Manor
names = constructors.set_index("constructorId")["name"]
lineage = lambda s: s.map(LINEAGE).fillna(s.map(names))
df["team"] = lineage(df["constructorId"])

cs = cstand_all.merge(races_all[["raceId", "year", "round"]], on="raceId", how="left", validate="many_to_one")
cs["team"] = lineage(cs["constructorId"])
# (a) same season, previous round
prev = cs[["year", "round", "constructorId", "position"]].rename(columns={"position": "constructor_rank_prev"})
prev["round"] += 1
n0 = len(df)
df = df.merge(prev, on=["year", "round", "constructorId"], how="left", validate="many_to_one")
# (b) round 1: final standings of the previous season, by lineage
last_round = cs.groupby("year")["round"].transform("max")
final = (cs[cs["round"] == last_round][["year", "team", "position"]]
         .assign(year=lambda d: d["year"] + 1).rename(columns={"position": "rank_prev_season"}))
df = df.merge(final, on=["year", "team"], how="left", validate="many_to_one")
from_prev_season = df["constructor_rank_prev"].isna() & (df["round"] == 1)
df.loc[from_prev_season, "constructor_rank_prev"] = df.loc[from_prev_season, "rank_prev_season"]
df = df.drop(columns="rank_prev_season"); assert len(df) == n0

print(f"constructor_rank_prev available: {df['constructor_rank_prev'].notna().mean():.1%} "
      f"(round 1 filled from previous season: {from_prev_season.sum() - df.loc[from_prev_season, 'constructor_rank_prev'].isna().sum()})")
display(df[df["constructor_rank_prev"].isna()].groupby(["year", "team"]).size().rename("rows without rank").to_frame())

df["team_tier"] = pd.cut(df["constructor_rank_prev"], [0, 3, 6, 20], labels=["top 3", "4th–6th", "7th+"]).astype(object)
df["team_tier"].value_counts()

constructor_rank_prev available: 99.9% (round 1 filled from previous season: 220)


,,rows without rank
year,team,
2015,Manor,2
2016,Haas F1 Team,2


team_tier
7th+       1878
top 3      1362
4th–6th    1362
Name: count, dtype: int64

**Reading.** The control variable is available for 99.9 % of rows (220 round-1 rows filled from the previous season).
The only 4 gaps are teams with **no previous standings**: Haas in its first race (round 1, 2016) and Manor at round 2 of 2015
(it did not take part in round 1). This is correct: there is no pre-race information about them, so they are not imputed.
`team_tier` groups the rank into three strength levels for stratified analysis in notebook 03.

## 8. Context variables
Driver name, circuit, era of technical regulations (2022 change of car concept, goal "closer racing"), sprint weekends, and the
number of races each circuit has in the period (per-circuit KPIs require a minimum sample).

In [9]:
drv = drivers.assign(driver=drivers["forename"] + " " + drivers["surname"])[["driverId", "driver", "code"]]
n0 = len(df)
df = (df.merge(drv, on="driverId", how="left", validate="many_to_one")
        .merge(circuits[["circuitId", "location", "country"]].rename(columns={"location": "circuit"}),
               on="circuitId", how="left", validate="many_to_one"))
assert len(df) == n0
df["era"] = np.where(df["year"] >= 2022, "2022–2024", "2014–2021")
df["sprint_weekend"] = df["raceId"].isin(sprint["raceId"])
races_per_circuit = df.groupby("circuit")["raceId"].nunique()
df["circuit_races"] = df["circuit"].map(races_per_circuit)
df["top10"], df["podium"], df["win"] = df["final_pos"] <= 10, df["final_pos"] <= 3, df["final_pos"] == 1
print(f"Circuits: {len(races_per_circuit)} | with >= 5 races in the period: {(races_per_circuit >= 5).sum()} "
      f"| sprint weekends: {df.loc[df['sprint_weekend'], 'raceId'].nunique()}")

Circuits: 32 | with >= 5 races in the period: 19 | sprint weekends: 18


**Five Cs · Consistent: does "grid" mean the same thing on every weekend?** On the sprint weekends of 2021–2022 the Saturday sprint
set the Grand Prix grid; from 2023 the qualifying session does again. We compare how often the grid differs from the qualifying order.

In [10]:
both = df[df["grid"].notna() & df["quali_pos"].notna()]
weekend = np.select([both["sprint_weekend"] & both["year"].between(2021, 2022), both["sprint_weekend"]],
                    ["sprint weekend 2021–2022", "sprint weekend 2023–2024"], "standard weekend")
(both.groupby(weekend)["grid_differs_quali"].agg(results="size", share_grid_differs_from_quali="mean")
     .round(3).rename_axis("weekend type"))

,results,share_grid_differs_from_quali
weekend type,,
sprint weekend 2021–2022,113,0.841
sprint weekend 2023–2024,228,0.272
standard weekend,4174,0.289


**Reading.** On standard weekends about 3 in 10 grids differ from the qualifying order (penalties), but on the 2021–2022 sprint
weekends more than 8 in 10 do, because the grid came from the sprint. "Grid" is therefore consistent as **the real starting order**,
while "qualifying → result" is not comparable on those weekends. This supports using the grid as the main measure, keeping the
qualifying position as a sensitivity check and flagging sprint weekends (`sprint_weekend`).

## 9. Final validation of the analytic table
Checks that must hold before any KPI is computed: unique key, valid ranges, flags consistent with the values, and a
missing-value profile (which gaps are **structural** and which would be errors).

In [11]:
COLS = ["raceId", "year", "round", "date", "name", "circuitId", "circuit", "country", "circuit_races", "era", "sprint_weekend",
        "driverId", "driver", "code", "constructorId", "team", "constructor_rank_prev", "team_tier",
        "grid", "pitlane_start", "grid_block", "quali_pos", "quali_best_s", "quali_gap_pct", "quali_outlier", "grid_differs_quali",
        "pos_lap1", "gained_lap1", "race_pace_gap_pct", "fastest_lap_rank",
        "final_pos", "position", "positions_gained", "finished", "classified", "status", "end_cause",
        "top10", "podium", "win", "points", "laps"]
at = df[COLS].sort_values(["year", "round", "final_pos"]).reset_index(drop=True)

checks = {
    "key (raceId, driverId) unique": not at.duplicated(["raceId", "driverId"]).any(),
    "grid in 1..26 when not NaN": at["grid"].dropna().between(1, 26).all(),
    "grid NaN only for pit-lane starts": (at["grid"].isna() == at["pitlane_start"]).all(),
    "one winner per race": (at.groupby("raceId")["win"].sum() == 1).all(),
    "final_pos 1..n per race": at.groupby("raceId")["final_pos"].apply(lambda s: s.max() == len(s)).all(),
    "positions_gained = grid - final_pos": np.allclose((at["grid"] - at["final_pos"]).dropna(), at["positions_gained"].dropna()),
    "quali gap >= 0": (at["quali_gap_pct"].dropna() >= 0).all(),
    "race pace gap >= 0": (at["race_pace_gap_pct"].dropna() >= 0).all(),
    "Belgium 2021 excluded": not at["raceId"].isin(not_raced).any(),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())

miss = at.isna().mean().mul(100).round(1)
display(miss[miss > 0].sort_values(ascending=False).to_frame("% missing"))
print(f"Analytic table: {len(at):,} rows x {at.shape[1]} columns | {at['raceId'].nunique()} races | "
      f"{at['driverId'].nunique()} drivers | {at['team'].nunique()} team lineages")

,passed
"key (raceId, driverId) unique",True
grid in 1..26 when not NaN,True
grid NaN only for pit-lane starts,True
one winner per race,True
final_pos 1..n per race,True
positions_gained = grid - final_pos,True
quali gap >= 0,True
race pace gap >= 0,True
Belgium 2021 excluded,True


,% missing
position,15.7
race_pace_gap_pct,11.3
gained_lap1,4.5
fastest_lap_rank,4.3
pos_lap1,3.0
positions_gained,1.7
grid,1.7
quali_best_s,1.5
quali_gap_pct,1.5
quali_pos,0.3


Analytic table: 4,606 rows x 42 columns | 227 races | 59 drivers | 12 team lineages


**Reading.** All checks pass. Every remaining missing value has a known, **structural** reason, so none is imputed:

| Column | Why it is missing |
|---|---|
| `position` | not classified (retired before 90 % of the distance) |
| `race_pace_gap_pct` | retired too early to measure pace (< 50 % of laps) |
| `fastest_lap_rank` | no timed lap (retired on lap 1 or did not start) |
| `grid`, `positions_gained` | pit-lane start (flagged in `pitlane_start`) |
| `quali_*` | no qualifying time set |
| `pos_lap1`, `gained_lap1` | did not complete lap 1 |
| `constructor_rank_prev` | new team in its first race |

## 10. Save outputs: analytic table, data dictionary and cleaning log

In [12]:
at.to_csv(PROC / "analytic_driver_race_2014_2024.csv", index=False)

DICT = {"raceId": "race id (Ergast)", "year": "season", "round": "round within season", "date": "race date", "name": "Grand Prix name",
 "circuitId": "circuit id", "circuit": "circuit location", "country": "circuit country", "circuit_races": "races held at that circuit in 2014–2024",
 "era": "technical regulations era (2014–2021 / 2022–2024)", "sprint_weekend": "weekend had a sprint race",
 "driverId": "driver id", "driver": "driver name", "code": "3-letter driver code", "constructorId": "team id (as entered)",
 "team": "team lineage (stable name across renames)", "constructor_rank_prev": "team championship position after the previous round (round 1: previous season final)",
 "team_tier": "top 3 / 4th–6th / 7th+ by constructor_rank_prev", "grid": "starting position (NaN = pit-lane start)",
 "pitlane_start": "started from the pit lane (raw grid = 0)", "grid_block": "grid group: 1–3, 4–6, 7–10, 11–15, 16+, pit lane",
 "quali_pos": "qualifying classification", "quali_best_s": "best of Q1/Q2/Q3 in seconds", "quali_gap_pct": "gap of best qualifying time to fastest in that race (%)",
 "quali_outlier": "quali gap > 7 %", "grid_differs_quali": "grid position differs from qualifying position",
 "pos_lap1": "position at the end of lap 1", "gained_lap1": "grid − pos_lap1 (+ = gained at the start)",
 "race_pace_gap_pct": "median representative lap vs fastest driver's median in that race (%)", "fastest_lap_rank": "rank of the driver's fastest lap",
 "final_pos": "final order (positionOrder, retirements at the bottom)", "position": "official classified position (NaN = not classified)",
 "positions_gained": "grid − final_pos (+ = gained)", "finished": "took the chequered flag (Finished / +n Laps)",
 "classified": "officially classified", "status": "raw finishing status", "end_cause": "finished / incident / mechanical / other",
 "top10": "finished in the points (top 10)", "podium": "top 3", "win": "won the race", "points": "championship points", "laps": "laps completed"}
dd = pd.DataFrame({"column": COLS, "dtype": [str(at[c].dtype) for c in COLS], "description": [DICT[c] for c in COLS],
                   "% missing": [round(100 * at[c].isna().mean(), 1) for c in COLS]})
dd.to_csv(PROC / "data_dictionary.csv", index=False)

for step, rule, ev in [
    ("3. Pit-lane starts", "grid = 0 -> grid NaN + pitlane_start flag (kept, not deleted)", "Notebook 01, part 5"),
    ("4. Qualifying join", "results -> qualifying, one-to-one; times parsed to seconds; gap % to fastest", "validate='one_to_one'"),
    ("5. Race pace", "Median lap excl. lap 1, in/out laps and laps > 107 % of race median; >= 50 % laps", "Section 6"),
    ("6. Team strength", "Constructor rank after previous round; round 1 = previous season final (by lineage)", "Avoids leakage"),
    ("7. Team lineage", "20 constructor ids -> 12 lineages", "W1 A.5")]:
    log(step, rule, len(at), len(at), ev)
pd.DataFrame(LOG).to_csv(PROC / "cleaning_log.csv", index=False)
display(pd.DataFrame(LOG))
print(sorted(os.listdir(PROC)))

,step,rule,rows before,rows after,evidence
0,1. Period,Keep seasons 2014–2024 (master filter on races...,26759,4626,"Notebook 01, parts 3–4"
1,2. Not-raced races,Exclude races with < 5 laps in lap_times,4626,4606,Belgium 2021: 1 lap behind the safety car
2,3. Pit-lane starts,grid = 0 -> grid NaN + pitlane_start flag (kep...,4606,4606,"Notebook 01, part 5"
3,4. Qualifying join,"results -> qualifying, one-to-one; times parse...",4606,4606,validate='one_to_one'
4,5. Race pace,"Median lap excl. lap 1, in/out laps and laps >...",4606,4606,Section 6
5,6. Team strength,Constructor rank after previous round; round 1...,4606,4606,Avoids leakage
6,7. Team lineage,20 constructor ids -> 12 lineages,4606,4606,W1 A.5


['analytic_driver_race_2014_2024.csv', 'cleaning_log.csv', 'data_dictionary.csv', 'kpi_IDC_by_circuit.csv', 'kpi_P_top10_by_grid_block.csv', 'kpi_pole_conversion_by_circuit.csv', 'kpi_summary_by_circuit.csv', 'race_level_kpis.csv']


## 11. Join register and Five Cs of the prepared table
Every join used to build the table, with its key, expected cardinality, coverage and effect on the row count
(values from the outputs above).

| Join | Key | Cardinality (validated) | Matched | Unmatched and why | Rows |
|---|---|---|---|---|---|
| results → races | raceId | many-to-one | 4,626 of 4,626 | none | 4,626 → 4,626 |
| results → status | statusId | many-to-one | all | none | unchanged |
| results → qualifying (part 5) | raceId + driverId | one-to-one | 4,590 of 4,606 | 16 drivers with no qualifying record (no time set) | 4,606 → 4,606 |
| lap_times → pit in/out laps (part 6) | raceId + driverId + lap | many-to-one | flag only | not applicable (laps without a stop) | 248,124 → 248,124 laps |
| → position after lap 1 (part 6) | raceId + driverId | one-to-one | 97.0 % | did not complete lap 1 | 4,606 → 4,606 |
| → race pace (part 6) | raceId + driverId | one-to-one | 88.7 % (99.9 % of finishers) | retired before half distance, by design | 4,606 → 4,606 |
| → constructor standings, previous round (part 7) | year + round − 1 + constructorId | many-to-one | 4,382 direct + 220 from the previous season | 4 rows: team with no previous standings | 4,606 → 4,606 |
| → drivers, circuits (part 8) | driverId, circuitId | many-to-one | all | none | 4,606 → 4,606 |

| C | Prepared table (2014–2024) |
|---|---|
| **Clean** | Belgium 2021 excluded (−20 rows); `grid = 0` → pit-lane flag (77 rows, kept); `fastest_lap_rank = 0` → NaN (217); 105 qualifying gaps > 7 % flagged, not deleted; 9 of 9 validation checks pass |
| **Consistent** | One engine formula and one points scale; "grid" = the real starting order (sprint weekends flagged, see part 8); positions used instead of points |
| **Conformed** | All joins validated with no duplicated or lost rows; 20 constructor ids grouped into 12 lineages; team strength taken from the previous round |
| **Current** | Seasons 2014–2024; the data end in 2024 |
| **Comprehensive** | 227 races, 59 drivers; team rank available for 99.9 % of rows, race pace for 99.9 % of finishers; **not observed:** weather, safety car, tyres, car setups and the reasons for penalties |